# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an example for loading and exploring the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

# If you want to see the full metadata structure, you may use:
# print(metadata.to_json())

## 2. Data Overview
Review the available record sets, fields, and their `@id`s.

The Croissant schema organizes data into *record sets*, each with associated *fields*. Each record set, field, and column can be referenced by its `@id`. We'll list these for exploration.

In [ ]:
# List all available record sets and their fields/columns by @id
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record sets:")

for rs in record_sets:
    print(f"\nRecord Set: {rs['@id']} (name: {rs.get('name','no name')})")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields/Columns @id in this record set:")
    for f in fields:
        print(f"    - {f['@id']} (name: {f.get('name','no name')})")
    if not fields:
        print("    (no fields listed)")

## 3. Data Extraction
We'll load the records from each record set into a pandas DataFrame for analysis and exploration.

Record set and field `@id`s were previously printed for reference.

In [ ]:
# Get all record set @ids
rs_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for rs_id in rs_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        num_records = len(df)
        print(f"Loaded record set '{rs_id}' ({num_records} records, columns: {df.columns.tolist()})")
    except Exception as e:
        print(f"Failed to load records for record set '{rs_id}': {e}")

In [ ]:
# Display the first rows of the first record set as an example (edit rs_ids[0] as appropriate)
if rs_ids:
    rs_id_example = rs_ids[0]
    print(f"Columns in {rs_id_example}:")
    print(dataframes[rs_id_example].columns.tolist())
    dataframes[rs_id_example].head()

## 4. Exploratory Data Analysis (EDA)
Let's select a numeric field (by `@id`) to filter, normalize, and group. If you are unsure of field names, print a record set's DataFrame columns as above.

The code below demonstrates filtering records by a threshold, normalizing a numeric column, and grouping.

In [ ]:
# Choose a record set to analyze (edit as appropriate)
record_set_id = rs_ids[0]  # Use the first record set for demonstration
df = dataframes[record_set_id]
print(f"Analyzing record set: {record_set_id}")

# Choose a numeric field to demonstrate filtering and normalization
print("Columns available:", df.columns.tolist())

# Example: Try to find a numeric field (e.g., field with 'count', 'weight', 'abundance', 'coverage' in name)
possible_numeric_fields = [c for c in df.columns if any(s in c.lower() for s in ['count', 'weight', 'abundance', 'coverage', 'peptide', 'mw', 'intensity', 'score', 'pvalue', 'id'])]
if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]  # Select the first matching column
else:
    numeric_field = None

print(f"Selected numeric field: {numeric_field}")

if numeric_field is not None:
    # Ensure it's numeric (may require conversion)
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = df[numeric_field].mean() if df[numeric_field].notnull().any() else 0
    # Filter records greater than threshold
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize
    col_norm = f"{numeric_field}_normalized"
    filtered_df[col_norm] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, col_norm]].head())

    # Try to group by another categorical field (such as 'accession', 'group', or 'description')
    group_fields = [c for c in df.columns if c != numeric_field and df[c].dtype == 'O']
    if group_fields:
        group_field = group_fields[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"\nGrouped mean {numeric_field} by '{group_field}':")
        print(grouped_df.head())
    else:
        print("No suitable categorical field found for grouping.")
else:
    print("No apparent numeric field detected for EDA. Please check record set column names.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll use matplotlib to plot a histogram of the numeric field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None and filtered_df[numeric_field].notnull().any():
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field} (filtered)")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()
else:
    print("No numeric field available for histogram plot.")

## 6. Conclusion
We successfully loaded, inspected, and performed basic analysis on the FAIR² mass spectrometry dataset using `mlcroissant`. We:

- Inspected metadata and record set structure using Croissant `@id`s
- Loaded data from each record set into pandas DataFrames
- Applied example filtering, normalization, and grouping on a selected numeric field
- Visualized data distributions

You can further extend this notebook to explore other fields, merge across record sets, or apply custom data processing and machine learning workflows.